# Logic Gym Worlds — Colab Quickstart

This notebook walks through the full pipeline:
1. Clone repo & install deps
2. Generate a dataset split
3. Inspect instances
4. Run evaluation (Claude/GPT-4)
5. Compute metrics

In [ ]:
# ── 1. Clone and install ────────────────────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/logic-gym-worlds.git
%cd logic-gym-worlds
!pip install -q -r requirements.txt

In [ ]:
# ── 2. Set API keys ─────────────────────────────────────────────────────────
import os
from google.colab import userdata  # Colab Secrets

# Store your keys in Colab Secrets (left sidebar 🔑) as:
#   ANTHROPIC_API_KEY or OPENAI_API_KEY
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
# ── 3. Quick smoke test — generate 20 instances ──────────────────────────────
!python experiments/scripts/generate_dataset.py --quick --output_dir data/processed

In [ ]:
# ── 4. Inspect one instance ──────────────────────────────────────────────────
import json

with open('data/processed/train.json') as f:
    instances = json.load(f)

inst = instances[0]
print('=== RULES ===')
for r in inst['rules']:
    print(' -', r['natural_language'])

print('\n=== FACTS ===')
for fact in inst['facts']:
    if fact['positive']:
        print(f" - {fact['args'][0]} is {inst['predicates'][fact['predicate']]}")

print('\n=== QUERIES ===')
for q in inst['queries'][:5]:
    print(f" [{q['gold_label'].upper():12}] {q['natural_language']}")

In [ ]:
# ── 5. Run evaluation (limit=5 for demo) ─────────────────────────────────────
!python experiments/scripts/run_evaluation.py \
    --instances data/processed/test_iid.json \
    --model claude-sonnet-4-6 \
    --style cot \
    --limit 5 \
    --output experiments/results/demo_claude_cot.json

In [ ]:
# ── 6. View results ───────────────────────────────────────────────────────────
with open('experiments/results/demo_claude_cot.json') as f:
    results = json.load(f)

print(f"Accuracy:            {results['accuracy']:.4f}")
print(f"Macro-F1:            {results['macro_f1']:.4f}")
print(f"Contradiction rate:  {results['avg_contradiction_rate']:.4f}")
print(f"Para invariance:     {results['avg_paraphrase_invariance']:.4f}")
print(f"\nPer-label accuracy:")
for label, acc in results['per_label_accuracy'].items():
    print(f"  {label:15}: {acc:.4f}")

In [ ]:
# ── 7. Full dataset generation (takes ~10min for 500 instances/split) ─────────
# Uncomment when ready:
# !python experiments/scripts/generate_dataset.py \
#     --num_instances 500 \
#     --domain animals \
#     --output_dir data/processed

In [ ]:
# ── 8. Run full eval on operator-OOD split ────────────────────────────────────
# !python experiments/scripts/run_evaluation.py \
#     --instances data/processed/test_operator_ood.json \
#     --model claude-sonnet-4-6 \
#     --style cot \
#     --output experiments/results/claude_operator_ood_cot.json

In [ ]:
# ── 9. Run tests ──────────────────────────────────────────────────────────────
!pytest tests/ -v